Part 1: Database Setup and Logic
1. Database Schema

In [1]:
CREATE TABLE Menu_Items (
    item_id SERIAL PRIMARY KEY,
    item_name VARCHAR(30) NOT NULL,
    item_price SMALLINT DEFAULT 0
);

SyntaxError: invalid syntax (1501968520.py, line 1)

2. menu_item.py

In [2]:
import psycopg2

class MenuItem:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def _execute_query(self, query, params):
        # Database connection details
        conn = psycopg2.connect(dbname="your_db", user="postgres", password="your_password", host="localhost")
        cur = conn.cursor()
        try:
            cur.execute(query, params)
            conn.commit()
            return True
        except Exception as e:
            print(f"Error: {e}")
            conn.rollback()
            return False
        finally:
            cur.close()
            conn.close()

    def save(self):
        query = "INSERT INTO Menu_Items (item_name, item_price) VALUES (%s, %s)"
        return self._execute_query(query, (self.name, self.price))

    def delete(self):
        query = "DELETE FROM Menu_Items WHERE item_name = %s"
        return self._execute_query(query, (self.name,))

    def update(self, new_name, new_price):
        query = "UPDATE Menu_Items SET item_name = %s, item_price = %s WHERE item_name = %s"
        success = self._execute_query(query, (new_name, new_price, self.name))
        if success:
            self.name = new_name
            self.price = new_price
        return success

In [3]:
import psycopg2

class MenuManager:
    @classmethod
    def _get_connection(cls):
        return psycopg2.connect(dbname="your_db", user="postgres", password="your_password", host="localhost")

    @classmethod
    def get_by_name(cls, name):
        conn = cls._get_connection()
        cur = conn.cursor()
        cur.execute("SELECT item_name, item_price FROM Menu_Items WHERE item_name = %s", (name,))
        item = cur.fetchone()
        conn.close()
        return item if item else None

    @classmethod
    def all_items(cls):
        conn = cls._get_connection()
        cur = conn.cursor()
        cur.execute("SELECT * FROM Menu_Items")
        items = cur.fetchall()
        conn.close()
        return items

In [ ]:
# import MenuItem
# from menu_manager import MenuManager

def show_user_menu():
    while True:
        print("\n--- Restaurant Manager ---")
        print("(V) View an Item")
        print("(A) Add an Item")
        print("(D) Delete an Item")
        print("(U) Update an Item")
        print("(S) Show the Menu")
        print("(E) Exit")

        choice = input("Choice: ").upper()

        if choice == 'V':
            name = input("Item name: ")
            item = MenuManager.get_by_name(name)
            print(f"Result: {item}" if item else "Item not found.")
        elif choice == 'A':
            add_item_to_menu()
        elif choice == 'D':
            remove_item_from_menu()
        elif choice == 'U':
            update_item_from_menu()
        elif choice == 'S':
            show_restaurant_menu()
        elif choice == 'E':
            show_restaurant_menu()
            print("Goodbye!")
            break

def add_item_to_menu():
    name = input("New item name: ")
    price = int(input("New item price: "))
    item = MenuItem(name, price)
    if item.save():
        print("Item was added successfully.")
    else:
        print("Error adding item.")

def remove_item_from_menu():
    name = input("Item name to remove: ")
    item = MenuItem(name, 0)
    if item.delete():
        print("Item was deleted successfully.")
    else:
        print("Error: Item could not be deleted.")

def update_item_from_menu():
    current_name = input("Current item name: ")
    new_name = input("New name: ")
    new_price = int(input("New price: "))
    item = MenuItem(current_name, 0)
    if item.update(new_name, new_price):
        print("Item was updated successfully.")
    else:
        print("Error updating item.")

def show_restaurant_menu():
    items = MenuManager.all_items()
    print("\n--- CURRENT MENU ---")
    for item in items:
        print(f"ID: {item[0]} | Name: {item[1]} | Price: ${item[2]}")

if __name__ == "__main__":
    show_user_menu()


--- Restaurant Manager ---
(V) View an Item
(A) Add an Item
(D) Delete an Item
(U) Update an Item
(S) Show the Menu
(E) Exit
Choice: A
